# Experiment comparison demo

Compare control vs variant using aggregates (requires DB).

In [ ]:
import os
from sqlalchemy import create_engine, select
from sqlalchemy.orm import sessionmaker

from app.models.daily_aggregate import DailyAggregateRow
from app.schemas.run_config import RunConfig
from app.services.simulation.engine import create_run_record, execute_simulation

url = os.environ.get("DATABASE_URL", "postgresql+psycopg://localhost:5432/pricing_simulator")
engine = create_engine(url)
Session = sessionmaker(bind=engine)
cfg = RunConfig(
    seed=101,
    horizon_days=60,
    baseline_end_day=20,
    experiment_start_day=21,
    customer_count=200,
    control_delivery_fee=2.99,
    variant_delivery_fee=1.99,
)
db = Session()
run_id = create_run_record(db, cfg)
execute_simulation(db, run_id, cfg)

rows = db.scalars(
    select(DailyAggregateRow)
    .where(DailyAggregateRow.run_id == run_id, DailyAggregateRow.phase == "experiment")
    .where(DailyAggregateRow.location_zone == "__all__")
).all()
from collections import defaultdict
tot = defaultdict(float)
for r in rows:
    if r.treatment:
        tot[r.treatment] += r.metrics.get("orders", 0)
print("orders by treatment", dict(tot))